In [ ]:
!pip install -q streamlit pyngrok

In [1]:
import os
from datasets import load_dataset, DatasetDict

DATASET_ID = "zacharielegault/Kermany2017-OCT"   # swap for a full OCT2017 mirror if you need all ~84k images
EXPECTED_CLASSES = ["CNV", "DME", "DRUSEN", "NORMAL"]

ds = load_dataset(DATASET_ID)
if not isinstance(ds, DatasetDict):           # single split -> wrap
    ds = DatasetDict({"train": ds})
print(ds)                                     # shows each split and its row count

# Export all splits as class-named JPGs
for split_name, split in ds.items():
    names = split.features["label"].names
    out_dir = f"export/{split_name}"
    os.makedirs(out_dir, exist_ok=True)
    for i, row in enumerate(split):
        row["image"].convert("RGB").save(f"{out_dir}/{names[row['label']]}_{i:04d}.jpg")
    print(split_name, "->", len(os.listdir(out_dir)), "images")

README.md:   0%|          | 0.00/2.79k [00:00<?, ?B/s]

data/train-00000-of-00016.parquet: reconstructing file:   0%|          |  0.00B /  369MB            

data/train-00000-of-00016.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00016.parquet: reconstructing file:   0%|          |  0.00B /  473MB            

data/train-00001-of-00016.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00016.parquet: reconstructing file:   0%|          |  0.00B /  462MB            

... (28 outputs left out)

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/108309 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 108309
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
})
train -> 108309 images
test -> 1000 images


In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/OCT_Final_Deployment"

os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder created:", PROJECT_DIR)

Project folder created: /content/drive/MyDrive/OCT_Final_Deployment


In [4]:
# Step 10.2 – Create app.py
%%writefile /content/drive/MyDrive/OCT_Final_Deployment/app.py

import streamlit as st
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os

# --------------------------------------------------
# PAGE CONFIGURATION
# --------------------------------------------------

st.set_page_config(
    page_title="OCT Retinal Disease Classification",
    page_icon="👁️",
    layout="wide"
)

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASS_NAMES = ["CNV", "DME", "DRUSEN", "NORMAL"]

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/Step6_Results"
)

MODEL_PATHS = {
    "ResNet18": os.path.join(CHECKPOINT_DIR, "step6_best_model.pt"),
    "MobileNetV2": os.path.join(CHECKPOINT_DIR, "step7_mobilenetv2_best.pt"),
    "VGG16": os.path.join(CHECKPOINT_DIR, "step7_vgg16_best.pt"),
    "DenseNet121": os.path.join(CHECKPOINT_DIR, "step7_densenet121_best.pt"),
}

# --------------------------------------------------
# MODEL ARCHITECTURES
# --------------------------------------------------

def build_model(model_name):

    if model_name == "ResNet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(model.fc.in_features, 4)
        )

    elif model_name == "MobileNetV2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(
            model.last_channel, 4
        )

    elif model_name == "VGG16":
        model = models.vgg16(weights=None)
        model.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        model.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 4)
        )

    elif model_name == "DenseNet121":
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(1024, 4)

    else:
        raise ValueError("Unsupported model")

    return model


# --------------------------------------------------
# MODEL LOADING
# --------------------------------------------------

@st.cache_resource
def load_model(model_name):

    path = MODEL_PATHS[model_name]

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Checkpoint not found: {path}"
        )

    model = build_model(model_name)

    checkpoint = torch.load(
        path,
        map_location=DEVICE,
        weights_only=False
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    # Remove DataParallel prefix if present
    state_dict = {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }

    model.load_state_dict(state_dict, strict=True)

    model.to(DEVICE)
    model.eval()

    return model


# --------------------------------------------------
# IMAGE PREPROCESSING
# --------------------------------------------------

preprocess = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# --------------------------------------------------
# HEADER
# --------------------------------------------------

st.title("👁️ OCT Retinal Disease Classification")
st.subheader("Deep Learning Based Medical Image Classification")

st.write(
    "Upload an OCT retinal image and select a trained "
    "deep learning model to obtain a predicted class."
)

st.divider()

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------

st.sidebar.header("Model Configuration")

selected_model = st.sidebar.selectbox(
    "Select Deep Learning Model",
    list(MODEL_PATHS.keys())
)

st.sidebar.write("Device:", DEVICE)

# --------------------------------------------------
# IMAGE UPLOAD
# --------------------------------------------------

uploaded_file = st.file_uploader(
    "Upload OCT Image",
    type=["png", "jpg", "jpeg", "bmp", "tif", "tiff"]
)

if uploaded_file is not None:

    image = Image.open(uploaded_file).convert("L")

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Original OCT Image")
        st.image(image, use_container_width=True)

    with col2:
        st.subheader("Prediction")

        if st.button("Classify OCT Image", type="primary"):

            try:
                with st.spinner("Loading model and predicting..."):

                    model = load_model(selected_model)

                    input_tensor = preprocess(image)
                    input_tensor = input_tensor.unsqueeze(0).to(DEVICE)

                    with torch.no_grad():
                        outputs = model(input_tensor)
                        probabilities = torch.softmax(outputs, dim=1)

                    probabilities = probabilities.cpu().numpy()[0]

                    predicted_index = int(np.argmax(probabilities))
                    predicted_class = CLASS_NAMES[predicted_index]
                    confidence = float(probabilities[predicted_index]) * 100

                st.success("Prediction completed")

                st.metric(
                    "Predicted Class",
                    predicted_class,
                    f"{confidence:.2f}% confidence score"
                )

                st.subheader("Class-wise Scores")

                chart_data = {
                    "Class": CLASS_NAMES,
                    "Score (%)": probabilities * 100
                }

                st.bar_chart(
                    chart_data,
                    x="Class",
                    y="Score (%)"
                )

                st.info(
                    "These scores are model outputs after softmax. "
                    "They are not clinically calibrated probabilities."
                )

            except Exception as e:
                st.error(f"Prediction failed: {e}")

else:
    st.info("Please upload an OCT image to begin.")

# --------------------------------------------------
# PROJECT INFORMATION
# --------------------------------------------------

st.divider()

st.header("About This Project")

st.write("""
This project compares deep learning architectures for
four-class OCT retinal image classification:

- CNV – Choroidal Neovascularization
- DME – Diabetic Macular Edema
- DRUSEN
- NORMAL

Models included:
ResNet18, MobileNetV2, VGG16 and DenseNet121.
""")

st.warning(
    "Research and educational demonstration only. "
    "This application is not a medical diagnostic device. "
    "Do not use its predictions to make clinical decisions."
)

st.caption("OCT Retinal Disease Classification | Step 10 Deployment")

Writing /content/drive/MyDrive/OCT_Final_Deployment/app.py


In [5]:
#Step 10.3 – Check that the file exists
import os

APP_PATH = os.path.join(PROJECT_DIR, "app.py")

print("App exists:", os.path.exists(APP_PATH))
print("App location:", APP_PATH)

App exists: True
App location: /content/drive/MyDrive/OCT_Final_Deployment/app.py


In [ ]:
# Step 10.4 – Install and launch Streamlit

In [ ]:
# 1 – Install dependencies

In [6]:
!pip install -q streamlit pyngrok

In [ ]:
# 2 – Start Streamlit
import subprocess
import time
import os

APP_PATH = "/content/drive/MyDrive/OCT_Final_Deployment/app.py"

process = subprocess.Popen(
    [
        "streamlit", "run", APP_PATH,
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit process started:", process.poll() is None)

Streamlit process started: True


In [7]:
# Step 10.5 – Launch using Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.1) ...
Setting up cloudflared (2026.11.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [10]:
import subprocess

try:
    process
except NameError:
    process = None

# Start Streamlit if it is not already running
if process is None or process.poll() is not None:
    process = subprocess.Popen(
        ["streamlit", "run", "app.py",
         "--server.port", "8501",
         "--server.headless", "true",
         "--server.address", "0.0.0.0"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    print("Streamlit started")
else:
    print("Streamlit already running")

Streamlit started


In [11]:
!npm install -q localtunnel
!curl -s ipv4.icanhazip.com      # this IP is the tunnel "password"
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 2s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸34.106.254.245
⠙^C


In [14]:
import os, shutil, glob

# 1. Remove the leftover full-dataset folders and the old zip
for d in glob.glob("/content/export/*/"):
    shutil.rmtree(d, ignore_errors=True)
if os.path.exists("/content/oct_1000.zip"):
    os.remove("/content/oct_1000.zip")

# 2. Check only the 1000 images remain
jpgs = glob.glob("/content/export/*.jpg")
print(len(jpgs), "images in /content/export")

# 3. Rebuild the zip and verify
shutil.make_archive("/content/oct_1000", "zip", "/content/export")
print(round(os.path.getsize("/content/oct_1000.zip")/1e6, 1), "MB")

1000 images in /content/export
64.6 MB


In [15]:
import os, zipfile
z = "/content/oct_1000.zip"
print(round(os.path.getsize(z)/1e6, 1), "MB")
print(len(zipfile.ZipFile(z).namelist()), "files")

64.6 MB
1000 files


In [18]:
pip install streamlit pillow

In [17]:
import streamlit as st
from pathlib import Path
from PIL import Image

ROOT = Path(r"D:\OCT_export")
files = sorted(ROOT.rglob("*.jpg"))

st.title("OCT Retinal Image Viewer")

classes = ["All", "CNV", "DME", "DRUSEN", "NORMAL"]
choice = st.sidebar.selectbox("Class", classes)
per_page = st.sidebar.slider("Images per page", 8, 48, 24, step=4)
cols_n = st.sidebar.slider("Columns", 2, 6, 4)

if choice != "All":
    files = [f for f in files if f.name.startswith(choice)]

pages = max(1, (len(files) + per_page - 1) // per_page)
page = st.sidebar.number_input("Page", 1, pages, 1)
st.caption(f"{len(files)} images | page {page} of {pages}")

batch = files[(page - 1) * per_page : page * per_page]
cols = st.columns(cols_n)
for i, f in enumerate(batch):
    with cols[i % cols_n]:
        st.image(Image.open(f), caption=f.name, use_container_width=True)

2026-11-21 03:17:47.531 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-21 03:17:47.588 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-11-21 03:17:47.589 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-21 03:17:47.590 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-21 03:17:47.592 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-21 03:17:47.594 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-21 03:17:47.595 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-21 03:17:47.596 Session state does not 

# New Section

In [9]:
#  2 – Start Streamlit and Cloudflare Tunnel
import subprocess
import time
import re

# Start Streamlit if it is not already running
if process.poll() is not None:
    process = subprocess.Popen(
        [
            "streamlit", "run",
            "/content/drive/MyDrive/OCT_Final_Deployment/app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true"
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT
    )

time.sleep(5)

# Start temporary Cloudflare tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("Waiting for your public dashboard link...")

for _ in range(60):
    line = tunnel.stdout.readline()

    if line:
        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:
            print("\nYOUR STREAMLIT DASHBOARD:")
            print(match.group(0))
            break

NameError: name 'process' is not defined

In [ ]:
import os

search_root = "/content/drive/MyDrive"

checkpoint_names = [
    "step6_best_model.pt",
    "step7_custom_cnn_best.pt",
    "step7_mobilenetv2_best.pt",
    "step7_vgg16_best.pt",
    "step7_densenet121_best.pt"
]

print("SEARCHING FOR OCT CHECKPOINTS")
print("=" * 60)

for root, dirs, files in os.walk(search_root):
    for name in checkpoint_names:
        if name in files:
            print("FOUND:", os.path.join(root, name))

SEARCHING FOR OCT CHECKPOINTS


In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/Colab Notebooks"

print("\nColab Notebooks exists:", os.path.exists(base))

if os.path.exists(base):
    print("\nFolders and files:")
    for item in os.listdir(base):
        print(item)

results = os.path.join(base, "Step6_Results")

print("\nStep6_Results exists:", os.path.exists(results))

if os.path.exists(results):
    print("\nCheckpoint files:")
    for item in os.listdir(results):
        print(item)

ValueError: Mountpoint must not already contain files

In [ ]:
from google.colab import drive
import os

MOUNT_PATH = "/content/gdrive"

os.makedirs(MOUNT_PATH, exist_ok=True)

drive.mount(MOUNT_PATH, force_remount=True)

print("Google Drive mounted:", os.path.exists(MOUNT_PATH))

Mounted at /content/gdrive
Google Drive mounted: True


In [ ]:
import os

search_root = "/content/gdrive/MyDrive"

checkpoint_names = [
    "step6_best_model.pt",
    "step7_custom_cnn_best.pt",
    "step7_mobilenetv2_best.pt",
    "step7_vgg16_best.pt",
    "step7_densenet121_best.pt"
]

for root, dirs, files in os.walk(search_root):
    for name in checkpoint_names:
        if name in files:
            print("FOUND:", os.path.join(root, name))

FOUND: /content/gdrive/MyDrive/step7_vgg16_best.pt
FOUND: /content/gdrive/MyDrive/Colab Notebooks/Step6_Results/step6_best_model.pt
FOUND: /content/gdrive/MyDrive/Colab Notebooks/Step6_Results/step7_custom_cnn_best.pt
FOUND: /content/gdrive/MyDrive/Colab Notebooks/Step6_Results/step7_mobilenetv2_best.pt
FOUND: /content/gdrive/MyDrive/Colab Notebooks/Step6_Results/step7_vgg16_best.pt
FOUND: /content/gdrive/MyDrive/Colab Notebooks/Step6_Results/step7_densenet121_best.pt


In [ ]:
import os

APP_PATH = "/content/drive/MyDrive/OCT_Final_Deployment/app.py"

# Use the existing app file
if not os.path.exists(APP_PATH):
    APP_PATH = "/content/gdrive/MyDrive/OCT_Final_Deployment/app.py"

print("App file:", APP_PATH)
print("Exists:", os.path.exists(APP_PATH))

assert os.path.exists(APP_PATH), "Cannot find app.py"

App file: /content/drive/MyDrive/OCT_Final_Deployment/app.py
Exists: True


In [ ]:
with open(APP_PATH, "r") as f:
    code = f.read()

code = code.replace(
    '/content/drive/MyDrive/Colab Notebooks/Step6_Results',
    '/content/gdrive/MyDrive/Colab Notebooks/Step6_Results'
)

with open(APP_PATH, "w") as f:
    f.write(code)

print("Checkpoint paths updated successfully!")

Checkpoint paths updated successfully!


In [ ]:
import os

CHECKPOINT_DIR = "/content/gdrive/MyDrive/Colab Notebooks/Step6_Results"

for filename in [
    "step6_best_model.pt",
    "step7_custom_cnn_best.pt",
    "step7_mobilenetv2_best.pt",
    "step7_vgg16_best.pt",
    "step7_densenet121_best.pt"
]:
    path = os.path.join(CHECKPOINT_DIR, filename)
    print(filename, ":", os.path.exists(path))

step6_best_model.pt : True
step7_custom_cnn_best.pt : True
step7_mobilenetv2_best.pt : True
step7_vgg16_best.pt : True
step7_densenet121_best.pt : True


In [ ]:
import subprocess
import time

# Stop old Streamlit process if available
try:
    process.terminate()
    process.wait(timeout=5)
except:
    pass

process = subprocess.Popen(
    [
        "streamlit", "run", APP_PATH,
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit restarted:", process.poll() is None)

Streamlit restarted: True
